In [1]:
import pandas as pd

df = pd.read_csv("moto_imu_sim/imu_data.csv")
print(df.dtypes, df.isna().sum())
g = df.groupby("ride_id")
print(g.size().describe())
print(g['label'].nunique().value_counts())

ride_id        str
t          float64
ax         float64
ay         float64
az         float64
gx         float64
gy         float64
gz         float64
label          str
dtype: object ride_id    0
t          0
ax         0
ay         0
az         0
gx         0
gy         0
gz         0
label      0
dtype: int64
count       36.000000
mean     59806.638889
std         24.780353
min      59762.000000
25%      59792.000000
50%      59809.500000
75%      59821.250000
max      59868.000000
dtype: float64
label
1    36
Name: count, dtype: int64


STEP 2: SAMPLING REGULARITY :Checking whether the time distribution, or the time between reading is constant

In [ ]:
dt = df.groupby("ride_id")["t"].diff().dropna()# groupby ride_id, but subtract time"t" consecutively i.e row 2 - row 1, row 3 - row 2 and so on. so is basically time difference dt
print(dt.describe())
print(dt.round(6).value_counts().head())
print("non-positive:", (dt <= 0).sum(), "| gaps >7.5ms:", (dt > 0.0075).sum())

count    2.153003e+06
mean     5.016166e-03
std      5.774236e-04
min      1.493000e-03
25%      4.857000e-03
50%      5.001000e-03
75%      5.144000e-03
max      5.548500e-02
Name: t, dtype: float64
t
0.005019    4179
0.004964    4171
0.004981    4139
0.005021    4128
0.005011    4111
Name: count, dtype: int64
non-positive: 0 | gaps >7.5ms: 5208


In [ ]:
df["dt"] = df.groupby("ride_id")["t"].diff()
df["gap"] = df["dt"] > 0.0075 # df["gap"] keeps the boolean value of dt col, if it's greater than 0.0075

In [5]:
print(df.groupby("label")["gap"].mean())

label
aggressive    0.002515
calm          0.002343
normal        0.002428
Name: gap, dtype: float64


essentially similar


STEP 3: CLIPPING AND VALUE RANGES

Reason: sensor have a max range (eg: +/- 16g). Anything beyond will be cutoff flat and values gather at exactly the max. Hides the agressiveness of the ride 

In [6]:
print(df[["ax", "ay", "az", "gx", "gy", "gz"]].agg(["min", "max"]))

           ax        ay        az          gx          gy         gz
min -1.380615 -1.071289 -2.388428 -112.884521 -108.123779 -53.344727
max  1.354492  1.078613  5.677490  114.471436   93.475342  50.140381


aggegerated the min and max of each col.

no sign of clipping neither min nor max saturate at the sensor max value


STEP 4: GRAVITY AXIS PER RIDE.(mount leak check)

checking how the sensor was mounted in the bike
and which axis the gravity is acting


In [8]:
m = df.groupby("ride_id")[["ax", "ay", "az"]].mean()
m["label"] = df.groupby("ride_id")["label"].first()
print(m.sort_values("label"))

               ax        ay        az       label
ride_id                                          
R002    -0.009381 -0.019232  1.047785  aggressive
R004    -0.048712  0.045065  1.090882  aggressive
R005    -0.098032 -0.062199  0.999784  aggressive
R006    -0.066109  0.037728  1.050211  aggressive
R007    -0.006551  0.007377  1.031298  aggressive
R032    -0.021686  0.016803  0.993761  aggressive
R031     0.018692  0.114509  1.080766  aggressive
R020    -0.086482  0.016941  1.039926  aggressive
R001    -0.041379 -0.011114  0.961665        calm
R033    -0.097293 -0.049270  0.989642        calm
R029     0.053839 -0.007548  1.093154        calm
R027     0.038558  0.004135  0.964199        calm
R021    -0.049948  0.094717  0.942698        calm
R019    -0.066460  0.043149  0.993900        calm
R035    -0.015543 -0.042931  0.992679        calm
R018    -0.000395 -0.094732  1.055009        calm
R014     0.013074 -0.094014  1.064845        calm
R013     0.007046  0.059542  1.001943        calm
